# Tutorial 17: Transfer learning across component classes

Tutorial 16 crossed geometry regimes inside one class. Now we cross an actual class
boundary:

- foundation source: `GeneralizedCapNInterdigital` (3,683 designs);
- target specialist: `CapNInterdigitalTee` (894 designs);
- shared physical target: mutual conductor capacitance.

This is a difficult test. The option dictionaries, shape families, and capacitance
scales differ. We compare a manually aligned parameter baseline with the universal v0
layout representation.

## Claims we will test

1. Raw design options do not form a reusable cross-class feature contract without
   manual schema alignment.
2. v0 makes one model architecture usable across classes.
3. Few-shot transfer can approach or match a full target specialist with fewer labels.
4. Zero-shot failure remains possible when source-target geometry similarity is low.

In [2]:
import json
import logging
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from huggingface_hub import hf_hub_download
from plotly.subplots import make_subplots

from squadds.layouts import (
    PartitionTransferStudy,
    SourceFeatureProjector,
    StaticEmbeddingClient,
    TransferRidgeRegressor,
    V0PartitionTransferStudy,
    canonical_design_id,
    compress_v0_embeddings,
    regression_scores,
)

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 20)
logging.getLogger("httpx").setLevel(logging.WARNING)

EMBEDDING_REVISION = os.getenv("SQUADDS_EMBEDDING_REVISION", "main")
DATABASE_REVISION = os.getenv("SQUADDS_DATABASE_REVISION", "main")
FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75]
SOURCE = "GeneralizedCapNInterdigital"
TARGET = "CapNInterdigitalTee"
REPRESENTATION_COLORS = {
    "shared parameters": "#D1495B",
    "v0 embeddings": "#00798C",
}

## 1. Build one supervised task across two classes

`north_to_south` and `top_to_bottom` use different pin vocabulary but represent the
same physical quantity: mutual capacitance between the two functional conductors.
Because values span almost two decades, the model predicts `log(1 + C/fF)`.

In [3]:
embedding_client = StaticEmbeddingClient(revision=EMBEDDING_REVISION)
embedding_catalogue = embedding_client.embeddings()


def parse_um(value):
    return float(str(value).replace("um", ""))


def load_class(filename, component_name, result_name):
    path = hf_hub_download(
        "SQuADDS/SQuADDS_DB",
        filename,
        repo_type="dataset",
        revision=DATABASE_REVISION,
    )
    with open(path, encoding="utf-8") as stream:
        rows = json.load(stream)
    records = []
    option_names = set()
    for row in rows:
        options = row["design"]["design_options"]
        option_names.update(options)
        records.append(
            {
                "design_id": canonical_design_id(component_name, options),
                "component_name": component_name,
                "finger_count": float(options["finger_count"]),
                "finger_length_um": parse_um(options["finger_length"]),
                "mutual_capacitance_fF": float(row["sim_results"][result_name]),
            }
        )
    frame = embedding_catalogue.loc[
        embedding_catalogue["component_name"] == component_name
    ].merge(pd.DataFrame(records), on=["design_id", "component_name"], validate="one_to_one")
    return frame, option_names


generalized, generalized_options = load_class(
    "coupler-GeneralizedCapNInterdigital-cap_matrix.json",
    SOURCE,
    "north_to_south",
)
capn, capn_options = load_class(
    "coupler-CapNInterdigitalTee-cap_matrix.json",
    TARGET,
    "top_to_bottom",
)
data = pd.concat([generalized, capn], ignore_index=True)
data["log_mutual_capacitance"] = np.log1p(data["mutual_capacitance_fF"])

schema_summary = pd.DataFrame(
    {
        "component": [SOURCE, TARGET],
        "design options": [len(generalized_options), len(capn_options)],
        "shared option names": [
            len(generalized_options & capn_options),
            len(generalized_options & capn_options),
        ],
        "designs": [len(generalized), len(capn)],
    }
)
schema_summary

,component,design options,shared option names,designs
0,GeneralizedCapNInterdigital,41,3,3683
1,CapNInterdigitalTee,11,3,894


In [4]:
# %% hide input
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Feature schemas do not align automatically", "The shared target shifts"],
)
fig.add_trace(
    go.Bar(
        x=schema_summary["component"],
        y=schema_summary["design options"],
        name="all option names",
        marker_color="#D1495B",
        text=schema_summary["design options"],
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=schema_summary["component"],
        y=schema_summary["shared option names"],
        name="shared names",
        marker_color="#E9C46A",
        text=schema_summary["shared option names"],
    ),
    row=1,
    col=1,
)
for component, color in [(SOURCE, "#00798C"), (TARGET, "#6A4C93")]:
    frame = data.loc[data["component_name"] == component]
    fig.add_trace(
        go.Histogram(
            x=frame["mutual_capacitance_fF"],
            name=component,
            marker_color=color,
            opacity=0.62,
            nbinsx=38,
        ),
        row=1,
        col=2,
    )
fig.update_layout(
    title="Cross-class transfer must bridge both schema and response shift",
    barmode="group",
    height=520,
    template="plotly_white",
)
fig.update_yaxes(title_text="option names", row=1, col=1)
fig.update_xaxes(title_text="mutual capacitance (fF)", row=1, col=2)
fig.update_yaxes(title_text="designs", row=1, col=2)
fig.show()

## 2. Put the class boundary in the full v0 catalogue

All four supported component families share the same 9,227-dimensional v0 contract.
For visualization only, we pool shape pixels, standardize globally, and project to two
principal directions. The supervised experiment below still uses source-only fitting.

In [5]:
all_embeddings = np.vstack(embedding_catalogue["embedding"]).astype(np.float32)
visual_features = compress_v0_embeddings(all_embeddings, pooled_shape_size=12)
visual_features -= visual_features.mean(axis=0, keepdims=True)
visual_scale = visual_features.std(axis=0, keepdims=True)
visual_scale[visual_scale < 1e-6] = 1
visual_features /= visual_scale
rng = np.random.default_rng(17)
sketch = visual_features @ rng.normal(
    0,
    1 / np.sqrt(32),
    size=(visual_features.shape[1], 32),
)
u, singular_values, _ = np.linalg.svd(sketch, full_matrices=False)
catalogue_projection = embedding_catalogue[["component_name", "source_id"]].copy()
catalogue_projection["axis_1"] = u[:, 0] * singular_values[0]
catalogue_projection["axis_2"] = u[:, 1] * singular_values[1]

In [6]:
# %% hide input
fig = px.scatter(
    catalogue_projection,
    x="axis_1",
    y="axis_2",
    color="component_name",
    render_mode="webgl",
    opacity=0.62,
    color_discrete_map={
        "GeneralizedCapNInterdigital": "#00798C",
        "CapNInterdigitalTee": "#D1495B",
        "CavityClawRouteMeander": "#E9C46A",
        "TransmonCross": "#6A4C93",
    },
    hover_data={"source_id": True, "axis_1": ":.2f", "axis_2": ":.2f"},
    title="One representation contract spans four component classes",
)
fig.update_traces(marker={"size": 5})
fig.update_layout(
    height=620,
    template="plotly_white",
    xaxis_title="randomized PCA sketch, axis 1",
    yaxis_title="randomized PCA sketch, axis 2",
)
fig.show()

## 3. Compare a best-effort schema baseline with v0

The parameter baseline receives the only two directly comparable geometry fields used
by both classes: finger count and finger length. We strengthen it with squared terms
and an interaction, then fit normalization on the source class only.

The v0 model receives the same compact 155-feature representation used in Tutorial 15.
Both models use the same ridge head, split, label fractions, and random seeds.

In [7]:
domains = data["component_name"].to_numpy()
targets = data["log_mutual_capacitance"].to_numpy()[:, None]
raw_shared = data[["finger_count", "finger_length_um"]].to_numpy(dtype=float)
shared_polynomial = np.column_stack(
    [
        raw_shared,
        raw_shared[:, 0] ** 2,
        raw_shared[:, 1] ** 2,
        raw_shared[:, 0] * raw_shared[:, 1],
    ]
)
source_mask = domains == SOURCE
shared_projector = SourceFeatureProjector().fit(shared_polynomial[source_mask])
shared_study = PartitionTransferStudy(
    shared_projector.transform(shared_polynomial),
    targets,
    domains,
    SOURCE,
    target_names=["log(1 + mutual capacitance/fF)"],
    alpha=10.0,
)
v0_study = V0PartitionTransferStudy(
    np.vstack(data["embedding"]).astype(np.float32),
    targets,
    domains,
    SOURCE,
    target_names=["log(1 + mutual capacitance/fF)"],
    pooled_shape_size=12,
    alpha=10.0,
)

pd.DataFrame(
    {
        "representation": ["shared parameters", "v0 embeddings"],
        "features": [shared_study.features.shape[1], v0_study.features.shape[1]],
        "manual schema alignment": [True, False],
        "mean target-to-source cosine": [np.nan, v0_study.domain_similarity().iloc[0]["mean"]],
    }
)

,representation,features,manual schema alignment,mean target-to-source cosine
0,shared parameters,5,True,NaN
1,v0 embeddings,155,False,0.082225


In [8]:
curve_frames = []
benchmark_frames = []
for representation, current_study in [
    ("shared parameters", shared_study),
    ("v0 embeddings", v0_study),
]:
    curve_frames.append(
        current_study.learning_curves(
            FRACTIONS,
            repeats=12,
            test_fraction=0.30,
            random_seed=17,
        ).assign(representation=representation)
    )
    benchmark_frames.append(
        current_study.dedicated_benchmarks(
            test_fraction=0.30,
            random_seed=17,
        ).assign(representation=representation)
    )

curves = pd.concat(curve_frames, ignore_index=True)
benchmarks = pd.concat(benchmark_frames, ignore_index=True)
macro = curves.loc[curves["target"] == "macro"]
summary = (
    macro.groupby(["representation", "method", "sample_fraction"], as_index=False)
    .agg(
        r2=("r2", "mean"),
        r2_lower=("r2", lambda values: values.quantile(0.10)),
        r2_upper=("r2", lambda values: values.quantile(0.90)),
        log_mae=("mae", "mean"),
        sample_size=("sample_size", "first"),
    )
)
full_scores = benchmarks.query("target == 'macro' and method == 'dedicated-full'")[
    ["representation", "r2", "mae"]
]
summary.head()

,representation,method,sample_fraction,r2,r2_lower,r2_upper,log_mae,sample_size
0,shared parameters,target-only,0.049521,0.770022,0.753490,0.795229,0.336828,31
1,shared parameters,target-only,0.100639,0.795321,0.772243,0.810443,0.320380,63
2,shared parameters,target-only,0.249201,0.843509,0.840777,0.847761,0.285153,156
3,shared parameters,target-only,0.500000,0.872435,0.867920,0.876277,0.262849,313
4,shared parameters,target-only,0.750799,0.888313,0.886859,0.889962,0.250535,470


In [9]:
# %% hide input
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Manually aligned shared parameters", "Universal v0 embeddings"],
    shared_yaxes=True,
)
for column, representation in enumerate(REPRESENTATION_COLORS, start=1):
    for method, dash in [("target-only", "dot"), ("transfer", "solid")]:
        frame = summary.loc[
            (summary["representation"] == representation)
            & (summary["method"] == method)
        ].sort_values("sample_fraction")
        color = REPRESENTATION_COLORS[representation]
        fig.add_trace(
            go.Scatter(
                x=100 * frame["sample_fraction"],
                y=frame["r2"],
                mode="lines+markers",
                name=method,
                legendgroup=method,
                showlegend=column == 1,
                line={"color": color, "width": 3, "dash": dash},
                marker={"size": 8, "symbol": "square" if method == "transfer" else "circle"},
                error_y={
                    "type": "data",
                    "symmetric": False,
                    "array": frame["r2_upper"] - frame["r2"],
                    "arrayminus": frame["r2"] - frame["r2_lower"],
                    "width": 3,
                },
                customdata=frame["sample_size"],
                hovertemplate=(
                    f"<b>{method}</b><br>labels=%{{customdata}} (%{{x:.1f}}%)"
                    "<br>held-out R2=%{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
    dedicated = full_scores.loc[
        full_scores["representation"] == representation,
        "r2",
    ].iloc[0]
    fig.add_hline(
        y=dedicated,
        line_dash="dash",
        line_color="#1F2937",
        annotation_text=f"full dedicated={dedicated:.3f}",
        row=1,
        col=column,
    )
fig.update_xaxes(title_text="labeled CapN target pool (%)")
fig.update_yaxes(title_text="held-out R2 on log mutual capacitance", row=1, col=1)
fig.update_layout(
    title="v0 transfer reaches the full specialist regime with a fraction of target labels",
    height=570,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

## 4. Ask the claim directly

For each label fraction we compare:

- transfer versus target-only at the same budget;
- transfer versus the representation's full dedicated specialist;
- v0 transfer versus the full manually aligned parameter specialist.

The third comparison is important: it tests whether universal geometry features beat a
larger bespoke model, not merely whether two ridge fits differ.

In [10]:
claim_table = summary.loc[summary["method"] == "transfer", [
    "representation",
    "sample_fraction",
    "sample_size",
    "r2",
    "log_mae",
]].copy()
claim_table = claim_table.merge(
    full_scores.rename(columns={"r2": "full_dedicated_r2", "mae": "full_dedicated_log_mae"}),
    on="representation",
)
shared_full_r2 = full_scores.loc[
    full_scores["representation"] == "shared parameters",
    "r2",
].iloc[0]
claim_table["fraction_percent"] = (100 * claim_table["sample_fraction"]).round(1)
claim_table["gap_to_own_full"] = claim_table["full_dedicated_r2"] - claim_table["r2"]
claim_table["gain_over_full_shared_specialist"] = claim_table["r2"] - shared_full_r2
claim_table.round(4)

,representation,sample_fraction,sample_size,r2,log_mae,full_dedicated_r2,full_dedicated_log_mae,fraction_percent,gap_to_own_full,gain_over_full_shared_specialist
0,shared parameters,0.0495,31,0.8482,0.2810,0.8955,0.2438,5.0,0.0473,-0.0473
1,shared parameters,0.1006,63,0.8641,0.2705,0.8955,0.2438,10.1,0.0315,-0.0315
2,shared parameters,0.2492,156,0.8846,0.2527,0.8955,0.2438,24.9,0.0110,-0.0110
3,shared parameters,0.5000,313,0.8957,0.2425,0.8955,0.2438,50.0,-0.0001,0.0001
4,shared parameters,0.7508,470,0.9029,0.2354,0.8955,0.2438,75.1,-0.0074,0.0074
5,v0 embeddings,0.0495,31,0.2274,0.2460,0.9670,0.1094,5.0,0.7396,-0.6681
6,v0 embeddings,0.1006,63,0.9544,0.1399,0.9670,0.1094,10.1,0.0127,0.0588
7,v0 embeddings,0.2492,156,0.9690,0.1196,0.9670,0.1094,24.9,-0.0020,0.0735
8,v0 embeddings,0.5000,313,0.9722,0.1120,0.9670,0.1094,50.0,-0.0052,0.0767
9,v0 embeddings,0.7508,470,0.9696,0.1119,0.9670,0.1094,75.1,-0.0026,0.0741


In [11]:
# %% hide input
fig = px.bar(
    claim_table,
    x="fraction_percent",
    y="gain_over_full_shared_specialist",
    color="representation",
    barmode="group",
    color_discrete_map=REPRESENTATION_COLORS,
    labels={
        "fraction_percent": "labeled CapN target pool (%)",
        "gain_over_full_shared_specialist": "R2 gain over full shared-parameter specialist",
    },
    title="A positive bar means fewer-label transfer beats the full schema baseline",
)
fig.add_hline(y=0, line_color="#1F2937", line_dash="dash")
fig.update_layout(height=500, template="plotly_white")
fig.show()

## 5. Convert the log prediction back to physical capacitance

R2 is computed in log space to handle the response range. For physical intuition, we
fit a 10% transfer model and compare it with the full v0 specialist after applying
`expm1`, then report MAE and MAPE in fF.

In [12]:
target_indices = np.flatnonzero(domains == TARGET)
source_indices = np.flatnonzero(domains == SOURCE)
target_order = np.random.default_rng(17).permutation(len(target_indices))
test_count = round(0.30 * len(target_indices))
local_test = target_order[:test_count]
local_pool = target_order[test_count:]
local_ten_percent = np.random.default_rng(1710).choice(
    local_pool,
    size=round(0.10 * len(local_pool)),
    replace=False,
)


def fitted_predictions(current_study, selected, *, prior=True):
    x_source = current_study.features[source_indices]
    y_source = current_study.targets[source_indices]
    x_target = current_study.features[target_indices]
    y_target = current_study.targets[target_indices]
    source_model = TransferRidgeRegressor(10.0).fit(x_source, y_source)
    model = TransferRidgeRegressor(10.0).fit(
        x_target[selected],
        y_target[selected],
        prior=source_model if prior else None,
    )
    return np.expm1(model.predict(x_target[local_test])[:, 0])


expected_fF = data.loc[domains == TARGET, "mutual_capacitance_fF"].to_numpy()[local_test]
prediction_sets = {
    "shared transfer: 10%": fitted_predictions(shared_study, local_ten_percent),
    "v0 transfer: 10%": fitted_predictions(v0_study, local_ten_percent),
    "v0 dedicated: 100%": fitted_predictions(v0_study, local_pool, prior=False),
}
physical_scores = []
physical_parity = []
for method, predictions in prediction_sets.items():
    score = regression_scores(expected_fF, predictions, ["mutual capacitance"]).iloc[-1]
    physical_scores.append(
        {
            "method": method,
            "R2 (fF)": score["r2"],
            "MAE (fF)": score["mae"],
            "MAPE (%)": score["mape_percent"],
        }
    )
    physical_parity.extend(
        {
            "method": method,
            "simulated": expected,
            "predicted": predicted,
        }
        for expected, predicted in zip(expected_fF, predictions)
    )
pd.DataFrame(physical_scores).round(3)

,method,R2 (fF),MAE (fF),MAPE (%)
0,shared transfer: 10%,0.690,3.972,49.793
1,v0 transfer: 10%,0.948,1.771,17.740
2,v0 dedicated: 100%,0.781,1.569,15.582


In [13]:
# %% hide input
parity = pd.DataFrame(physical_parity)
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=list(prediction_sets),
    shared_xaxes=True,
    shared_yaxes=True,
)
for column, method in enumerate(prediction_sets, start=1):
    frame = parity.loc[parity["method"] == method]
    bounds = [
        min(frame["simulated"].min(), frame["predicted"].min()),
        max(frame["simulated"].max(), frame["predicted"].max()),
    ]
    fig.add_trace(
        go.Scattergl(
            x=frame["simulated"],
            y=frame["predicted"],
            mode="markers",
            marker={
                "size": 5,
                "opacity": 0.55,
                "color": "#00798C" if method.startswith("v0") else "#D1495B",
            },
            hovertemplate="simulated=%{x:.2f} fF<br>predicted=%{y:.2f} fF<extra></extra>",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
    fig.add_trace(
        go.Scatter(
            x=bounds,
            y=bounds,
            mode="lines",
            line={"dash": "dash", "color": "#9CA3AF"},
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
fig.update_xaxes(title_text="simulated mutual capacitance (fF)")
fig.update_yaxes(title_text="predicted mutual capacitance (fF)", row=1, col=1)
fig.update_layout(
    title="The cross-class result remains meaningful in physical units",
    height=530,
    template="plotly_white",
)
fig.show()

## 6. Claim audit and scope

The data support a nuanced foundation-model claim:

- **Without v0:** the two option dictionaries are incompatible. Even after manually
  selecting two shared fields and adding polynomial terms, the full specialist leaves
  substantial variance unexplained.
- **With v0:** the same feature contract and model head run unchanged across classes.
  Around 10% target labels, transfer enters the full-specialist performance regime;
  around 25%, it can match the full v0 dedicated benchmark in this split.
- **Transfer matters:** it improves the low-label v0 fit, but the gain shrinks as target
  labels dominate.
- **Five percent is too small here:** 31 target labels do not reliably constrain the
  155-feature v0 head. At 10%, the transfer result stabilizes and overtakes the full
  manually aligned specialist.
- **Zero-shot is not enough here:** mean cross-class cosine is low and the capacitance
  scale shifts. A foundation representation enables efficient calibration; it does not
  erase physics or domain shift.

Only the two NCap classes share a directly comparable supervised target today. Cavity
and Transmon layouts still participate in the same embedding space, but a supervised
cross-family study involving them needs a common target or a multi-task output head.